# Notebook 01 - Datenerhebung

**Forschungsfrage:** Welche Faktoren beeinflussen den Mietpreis in der Schweiz?

- Web Scraping (Bonuspunkt)
- OOP: ImmobilienScraper + Inserat-Klassen
- Regex-Parsing Rohstrings -> numerische Werte
- Python Datenstrukturen: List, Dict, Set, Tuple
- SQLite-Datenbank + SQL-Queries (Bonuspunkt)

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd, re
from pathlib import Path
from src.scraper import ImmobilienScraper
from src.database import ImmobilienDB
from src.models import Inserat
print('Imports OK')

## 1. Daten sammeln via Web Scraper

In [ ]:
scraper = ImmobilienScraper(delay_sek=1.5)
staedte = ['zuerich','bern','basel','genf','luzern','lausanne','winterthur','lugano']
df_roh = scraper.scrape_alle_staedte(staedte=staedte, n_beispiel=50)
print(f'Rohdatensatz: {df_roh.shape[0]} x {df_roh.shape[1]}')
df_roh.head()

## 2. Regex-Parsing Demo
Zeigt wie Regex Rohstrings in numerische Werte umwandelt.

In [ ]:
rohstrings = [
    ("CHF 2'450.- / Monat", 'preis'),
    ("1 800 CHF/Mt.", 'preis'),
    ('85 m2', 'flaeche'),
    ('120m2', 'flaeche'),
    ('3.5 Zimmer', 'zimmer'),
    ('4-Zi.', 'zimmer'),
    ('8001 Zuerich, ZH', 'ort'),
    ('3011 Bern', 'ort'),
]
muster = {
    'preis':   (r"['\s]", r"(\d+(?:\.\d+)?)"),
    'flaeche': (None,    r"(\d+(?:[.,]\d+)?)\s*m[2]?"),
    'zimmer':  (None,    r"(\d+(?:[.,]\d+)?)"),
    'ort':     (None,    r"(\d{4})\s+([^\,]+)"),
}
print('Regex-Parsing Demo:')
print('-' * 50)
for rohstring, typ in rohstrings:
    b, pat = muster[typ]
    s = re.sub(b, '', rohstring) if b else rohstring
    m = re.search(pat, s)
    print(f'  {rohstring:30} -> {m.group(1) if m else "kein Treffer"}')

## 3. Python Built-in Datenstrukturen

In [ ]:
# LIST
staedte_liste = list(df_roh['stadt'].dropna())
print(f'List (5): {staedte_liste[:5]}')
# SET
einzigartige = set(staedte_liste)
print(f'Set: {sorted(einzigartige)}')
# DICT
preise_dict = {s: round(df_roh[df_roh['stadt']==s]['preis_chf'].mean(),0)
               for s in einzigartige if len(df_roh[df_roh['stadt']==s]) > 0}
preise_dict = dict(sorted(preise_dict.items(), key=lambda x: x[1], reverse=True))
print('Dict (Preis pro Stadt):')
for s,p in preise_dict.items(): print(f'  {s:15} CHF {p:,.0f}')
# TUPLE
spanne = (df_roh['preis_chf'].min(), df_roh['preis_chf'].max())
print(f'Tuple Preisspanne: CHF {spanne[0]:,.0f} - {spanne[1]:,.0f}')

## 4. Datenbank + SQL-Abfragen

In [ ]:
db = ImmobilienDB()
db.kantone_befuellen()
inserate = []
for _, z in df_roh.iterrows():
    ins = Inserat(titel=str(z.get('titel','')), preis_raw=str(z.get('preis_chf','')),
        flaeche_raw=str(z.get('flaeche_m2','')), zimmer_raw=str(z.get('zimmer_anzahl','')),
        ort_raw=f"{z.get('plz','')} {z.get('stadt','')}", url=str(z.get('url','')))
    inserate.append(ins)
db.bulk_speichern(inserate)
print(db)

In [ ]:
# SQL GROUP BY - AVG, MIN, MAX, COUNT
print('SQL-Abfrage: Preisstatistik pro Stadt')
db.preisstatistik_pro_stadt()

## 5. Daten speichern

In [ ]:
df_final = db.alle_laden()
Path('../data').mkdir(exist_ok=True)
df_final.to_csv('../data/inserate_roh.csv', index=False)
print(f'Gespeichert: {len(df_final)} Zeilen')
df_final.describe().round(1)